# Import Library

In [309]:
import pandas as pd
import re
import nltk
import joblib

from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Data

In [310]:
df = pd.read_excel("jamu206.xlsx")

df

,NO,NAMA JAMU,KHASIAT,KANDUNGAN,ATURAN MINUM,EFEK SAMPING,JENIS,PRODUSEN,LOKASI PEMASARAN,LOKASI PRODUKSI,KABUPATEN,PERIZINAN
0,1,Galian Rapat Wangi,"mengurangi bau badan, mengurangi bau tidak sed...","kunci, kunyit, pinang muda parabes",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
1,2,Empot-empot legit,"Mengurangi lendir berlebih, mengatasi keputiha...","kunci, kunyit, pinang muda parabes",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
2,3,galian singset,"mengurangi lemak dalam tubuh, menambah nafsu m...","kunyit, daun sirih, delima putih",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
3,4,jamu kecantikan,"perawatan khusus remaja putri, mengurangi bau ...","daun sirih, kunyit, kulit manggis, kunci, pina...",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
4,5,jamu galian montok,untuk menambah nafsu makan,"temulawak , jahe",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
...,...,...,...,...,...,...,...,...,...,...,...,...
201,202,galian rapet,rapat mencengkram saat berhubungan intim.hingg...,NaN,3 x sehari @5 pil sekali minum,NaN,pil,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM
202,203,jamu dalimah putih,dan selalu awet muda,NaN,3 x sehari @ 5 biji,NaN,pil,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM
203,204,GALIAN SEHAT / MONTOK,Pil jamu ramuan Madura untuk wanita / pria yan...,"Rimpang Kunyit,Rimpang Temulawak, Rimpang Leng...",Minum 3 x 5 pil / hari.,NaN,pil,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM
204,205,minyak zaitun,Mangencangkan dan menghaluskan kulit muka dan ...,NaN,Oleskan minyak zaitun secukupnya bagian kulit/...,NaN,cair,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM


# Info Dataset

In [311]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   NO                206 non-null    int64 
 1   NAMA JAMU         206 non-null    object
 2   KHASIAT           202 non-null    object
 3   KANDUNGAN         124 non-null    object
 4   ATURAN MINUM      113 non-null    object
 5   EFEK SAMPING      158 non-null    object
 6   JENIS             178 non-null    object
 7   PRODUSEN          206 non-null    object
 8   LOKASI PEMASARAN  198 non-null    object
 9   LOKASI PRODUKSI   183 non-null    object
 10  KABUPATEN         203 non-null    object
 11  PERIZINAN         119 non-null    object
dtypes: int64(1), object(11)
memory usage: 19.4+ KB
None


# Missing value

In [312]:
print(df.isnull().sum())

NO                   0
NAMA JAMU            0
KHASIAT              4
KANDUNGAN           82
ATURAN MINUM        93
EFEK SAMPING        48
JENIS               28
PRODUSEN             0
LOKASI PEMASARAN     8
LOKASI PRODUKSI     23
KABUPATEN            3
PERIZINAN           87
dtype: int64


In [313]:
df = df.dropna(subset=['KHASIAT'])

In [314]:
print("Jumlah data setelah drop:", len(df))
print("Sisa missing KHASIAT:", df['KHASIAT'].isnull().sum())

Jumlah data setelah drop: 202
Sisa missing KHASIAT: 0


# Preprocessing

In [315]:
import re
import string

stop_words = set(stopwords.words('indonesian'))

def preprocess(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

In [316]:
df['clean'] = df['KHASIAT'].apply(preprocess)

df[['KHASIAT', 'clean']]

/tmp/ipykernel_11088/351056614.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['clean'] = df['KHASIAT'].apply(preprocess)


,KHASIAT,clean
0,"mengurangi bau badan, mengurangi bau tidak sed...",mengurangi bau badan mengurangi bau sedap rape...
1,"Mengurangi lendir berlebih, mengatasi keputiha...",mengurangi lendir berlebih mengatasi keputihan...
2,"mengurangi lemak dalam tubuh, menambah nafsu m...",mengurangi lemak tubuh menambah nafsu makan me...
3,"perawatan khusus remaja putri, mengurangi bau ...",perawatan khusus remaja putri mengurangi bau b...
4,untuk menambah nafsu makan,menambah nafsu makan
...,...,...
201,rapat mencengkram saat berhubungan intim.hingg...,rapat mencengkram berhubungan intimhingga mele...
202,dan selalu awet muda,awet muda
203,Pil jamu ramuan Madura untuk wanita / pria yan...,pil jamu ramuan madura wanita pria nafsu makan...
204,Mangencangkan dan menghaluskan kulit muka dan ...,mangencangkan menghaluskan kulit muka tubuh ku...


## Pembuatan Label

In [317]:
def labeling(text):
    text = str(text).lower()

    if any(k in text for k in [
        'stamina','tenaga','energi','lelah','capek','letih','lesu','loyo',
        'ejakulasi','perkasa','kejantanan','sperma','vitalitas','daya tahan',
        'kuat','kebugaran','fit','tidak mudah capek','menambah tenaga',
        'bab','buang air besar','nafsu makan','maag','asam lambung',
        'lemak','diet','kolesterol','berat badan','melangsingkan',
        'pelangsing','gemuk','kurus','metabolisme','pencernaan',
        'kembung','mual','perut','lambung', 'pegal','pegal linu','linu',
        'nyeri','rematik','encok', 'peredaran darah','sirkulasi darah','sendi','otot',
        'nyeri sendi','nyeri otot','sakit pinggang', 'demam','flu','batuk','pilek',
        'diabetes','kencing manis', 'darah tinggi','hipertensi','imun','daya tahan tubuh',
        'asma','ginjal','jantung','kolesterol tinggi','radang',
        'infeksi','penyakit','tipes','liver','hepatitis'
    ]):
        return 'stamina'

    elif any(k in text for k in [
        'keputih','keputihan','vagina','miss v','rahim','haid','menstruasi',
        'asi','kandungan','estrogen','organ intim','lendir','keputihan berlebih',
        'gatal kewanitaan','bau kewanitaan','cairan','basah','keset','rapet',
        'otot rahim','nifas','melahirkan','payudara','hormonal','serviks'
    ]):
        return 'kesehatan_wanita'

    elif any(k in text for k in [
        'kulit','wajah','jerawat','komedo','cerah','halus','glowing',
        'anti aging','keriput','flek','pori','cantik','awet muda',
        'putih','bersih','lembut','kencang','menghaluskan','memutihkan',
        'perawatan tubuh','masker','bekas jerawat','noda hitam','muka',
        'gatal','gatal-gatal','alergi','jamur','infeksi kulit',
        'koreng','eksim','kudis','panu','ruam','iritasi','kulit sensitif',
        'bau badan','bau mulut','bau ketiak','bau tidak sedap',
        'wangi','harum','aroma','segar','pengharum','keringat',
        'bau','mengharumkan','menyegarkan'
    ]):
        return 'perawatan'

    else:
        return 'lainnya'

In [318]:
df['label'] = df['clean'].apply(labeling)

/tmp/ipykernel_11088/2066145728.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['clean'].apply(labeling)


In [319]:
df['label'].value_counts()

,count
label,
stamina,113
kesehatan_wanita,48
lainnya,21
perawatan,20


# Split Data

In [320]:
X = df['clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

In [321]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

In [322]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train_vec, y_train)

In [323]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_res, y_train_res)

MultinomialNB()

In [324]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

                  precision    recall  f1-score   support

kesehatan_wanita       0.71      1.00      0.83         5
         lainnya       0.00      0.00      0.00         2
       perawatan       0.33      0.50      0.40         2
         stamina       0.90      0.75      0.82        12

        accuracy                           0.71        21
       macro avg       0.49      0.56      0.51        21
    weighted avg       0.72      0.71      0.70        21



In [325]:
joblib.dump(tfidf, "tfidf.pkl")
joblib.dump(model, "model.pkl")

['model.pkl']

# Prediksi

In [326]:
import joblib
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

model = joblib.load("model.pkl")
tfidf = joblib.load("tfidf.pkl")

df = pd.read_excel("jamu206.xlsx")

In [327]:
df['clean'] = df['KHASIAT'].apply(preprocess)

In [328]:
tfidf_matrix = tfidf.transform(df['clean'])

In [329]:
def rekomendasi_jamu(text, top_n=3):
    clean_text = preprocess(text)

    input_vec = tfidf.transform([clean_text])

    similarity = cosine_similarity(input_vec, tfidf_matrix)

    top_idx = similarity[0].argsort()[::-1][:top_n]

    hasil = []

    for i in top_idx:
        hasil.append({
            "nama_jamu": df.iloc[i]['NAMA JAMU'],
            "khasiat": df.iloc[i]['KHASIAT'],
            "skor": round(similarity[0][i], 4)
        })

    return hasil

In [330]:
input_text = "saya sering pegal dan nyeri badan"

hasil = rekomendasi_jamu(input_text, top_n=10)

for i, h in enumerate(hasil, 1):
    print(f"Rekomendasi {i}")
    print("Nama Jamu :", h['nama_jamu'])
    print("Khasiat   :", h['khasiat'])
    print("Skor      :", h['skor'])
    print("-"*40)

Rekomendasi 1
Nama Jamu : Bom
Khasiat   : Untuk kuat berhubungan, dan Pegal - pegal
Skor      : 0.6232
----------------------------------------
Rekomendasi 2
Nama Jamu : Jaselang
Khasiat   : Menambah stamina, pegal linu, capek-capek, mengahangatkan badan dll
Skor      : 0.3561
----------------------------------------
Rekomendasi 3
Nama Jamu : Macho
Khasiat   : Menambah stamina dan ereksi, meningkatkan libido, mengobati pegal linu, capek-capek, dan menghangatkan badan
Skor      : 0.3101
----------------------------------------
Rekomendasi 4
Nama Jamu : sehat lelaki
Khasiat   : menambah semangat stamina dan kesehatan, menguatkan dan menghangatkan badan, pegal linu otot kaku dan lesu
Skor      : 0.3003
----------------------------------------
Rekomendasi 5
Nama Jamu : Jamu Sakit Pinggang
Khasiat   : meredakan nyeri pinggang dan menambah stamina
Skor      : 0.2912
----------------------------------------
Rekomendasi 6
Nama Jamu : jamu terlambat bulan (lancar darah)
Khasiat   : mengurangi n